In [5]:

import pandas as pd
import mysql.connector

file_path = r"C:\Users\rglra\Desktop\Zepto_SQL\olist_order_reviews_dataset_utf8.csv"

df = pd.read_csv(file_path, encoding="utf-8")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecommerce"
)
cursor = conn.cursor()

# Get order IDs that exist in your MySQL database
cursor.execute("SELECT order_id FROM orders")
valid_order_ids = {row[0] for row in cursor.fetchall()}

# Keep only reviews linked to existing orders
df = df[df["order_id"].isin(valid_order_ids)].copy()

# Replace pandas missing values with SQL NULL
df = df.astype(object).where(pd.notna(df), None)

sql = """
INSERT INTO reviews (
    review_id, order_id, review_score,
    review_comment_title, review_comment_message,
    review_creation_date, review_answer_timestamp
) VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

rows = list(df.itertuples(index=False, name=None))

try:
    for start in range(0, len(rows), 200):
        cursor.executemany(sql, rows[start:start + 200])
        conn.commit()

    print("Reviews imported:", len(rows))

except mysql.connector.IntegrityError as e:
    conn.rollback()
    print("Import error:", e)
    print("If reviews already exist, check duplicate review IDs.")

finally:
    cursor.close()
    conn.close()


Reviews imported: 1932
